# AlignTac students on Colab
Bundle on Drive: `MyDrive/aligntac_bundle/` (`01_code.tar.zst`, `02_real_core.tar.zst`, `03_sim_subset.tar.zst`, `setup.sh`). Run folders are synced to `MyDrive/aligntac_runs/` every 20 min, so a disconnected session resumes from the last synced epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE='/content/drive/MyDrive/aligntac_bundle'
SYNC='/content/drive/MyDrive/aligntac_runs'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc; df -h /content | tail -1

In [ ]:
# extract to the local disk (about 10-20 min the first time in a session) and map /media/hdd/ihsuan to it
!bash $BUNDLE/setup.sh $BUNDLE /content/aligntac
%cd /media/hdd/ihsuan/VisTacFusion/AlignTac
import os; os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'

In [ ]:
# optional: check the bundle (datasets load, no dangling links, CPU forward pass)
!python portable/verify_data.py

In [ ]:
# speed and memory of this runtime (about 3 min): prints steps/s and hours per student
!python portable/run_jobs.py --bench

In [ ]:
# train + evaluate. --parallel: 1 on T4 if the CPU is the limit, 2 on T4 (16 GB), 3-4 on L4, up to 6 on A100; --workers 2-4 when several jobs share few CPU cores
!python portable/run_jobs.py --jobs portable/jobs_loo3.tsv --parallel 2 --workers 3 --sync-to $SYNC

In [ ]:
# results (also copied to MyDrive/aligntac_runs/_results)
import json
for f in ['ablation/pilot_robosoft/loo3_results.json','ablation/pilot_robosoft/rss2_testA_results.json']:
    d=json.load(open(f))
    for k,v in d.items():
        if k.startswith('rss2_') and k.endswith(('/loo3test','/testA')): print(k, round(v['c_mae'],3), round(v['iou'],3))